# FULL TRANSFORMER ARCHITECTURE

In [2]:
import tensorflow as tf

In [3]:
tf.random.set_seed(42)

# Dataset

In [4]:
sentences = [
    "I love machine learning models",
    "Transformers understand human language well",
    "Deep learning models learn patterns",
    "Attention helps models understand context"
]

In [5]:
seq_len = 5
d_model = 8

In [6]:
import numpy as np
from tensorflow.keras.layers import TextVectorization, Embedding, MultiHeadAttention


# Tokenizer

In [7]:
tokenizer = TextVectorization(
    output_mode ="int",
    output_sequence_length = seq_len
)

tokenizer.adapt(sentences)

In [8]:
token_ids = tokenizer(tf.constant(sentences))

In [9]:
token_ids.shape

TensorShape([4, 5])

In [10]:
vocab_size = len(tokenizer.get_vocabulary())

In [11]:
print(vocab_size)

18


# Embedding

In [12]:
embedding = Embedding(
    input_dim = vocab_size,
    output_dim = d_model,
)

In [13]:
embeddings = embedding(token_ids)

In [14]:
print("\nEmbeddings shape:", embeddings.shape)


Embeddings shape: (4, 5, 8)


# Positional Encoding

In [15]:
def positional_encoding(seq_len,d_model):
  pos = np.arange(seq_len)[:,np.newaxis]
  i = np.arange(d_model)[np.newaxis,:]
  angle_rate = 1/np.power(
      10000,
      (2*(i//1))/np.float32(d_model)
  )
  angle_rads = pos*angle_rate

  # sine
  angle_rads[:,0::2] = np.sin(
      angle_rads[:,0::2]
  )

  # cos
  angle_rads[:,1::2] = np.cos(
      angle_rads[:,1::2]
  )

  return tf.cast(angle_rads[np.newaxis,...],
                 dtype = tf.float32)


In [16]:
pos_encoding = positional_encoding(seq_len, d_model)

In [17]:
print("shape of positional encoding",pos_encoding.shape)

shape of positional encoding (1, 5, 8)


In [18]:
final_input = embeddings + pos_encoding

In [19]:
print("after add of embedding with positional encoding ",final_input.shape)

after add of embedding with positional encoding  (4, 5, 8)


# Encoder Architecture start ( the input is ready to go in the encoder architecture)

In [20]:
multi_head = 2
key_dim = 4

MUltihead - attention

In [21]:
multi_head_attention = MultiHeadAttention(
    num_heads = multi_head,
    key_dim = key_dim
)

In [22]:
output,attention_scores = multi_head_attention(
    query = final_input,
    value = final_input,
    key = final_input,
    return_attention_scores = True
)

In [23]:
print("output shape",output.shape)

output shape (4, 5, 8)


In [24]:
print("attention score shape",attention_scores.shape)

attention score shape (4, 2, 5, 5)


In [25]:
input_shape = final_input + output

In [26]:
print(input_shape.shape)

(4, 5, 8)


Add & Norm

In [27]:
add_norm1 = tf.keras.layers.LayerNormalization()
add_norm1 = add_norm1(input_shape)

In [28]:
print(add_norm1.shape)

(4, 5, 8)


Feed Forward Neural Network

In [29]:
ffnn = tf.keras.Sequential([
    tf.keras.layers.Dense(128,activation = "relu"),
    tf.keras.layers.Dense(d_model)
])

In [30]:
ffnn_output = ffnn(add_norm1)

In [31]:
print(ffnn_output.shape)

(4, 5, 8)


Add & Norm

In [32]:
add_norm2 = tf.keras.layers.LayerNormalization()
add_norm2 = add_norm2(add_norm1 + ffnn_output)

In [33]:
print(add_norm2.shape)

(4, 5, 8)


In [37]:
encoder_output = add_norm2

In [38]:
print("the input for decoder architecture is ",encoder_output.shape)

the input for decoder architecture is  (4, 5, 8)


# Encoder Architecture is End ( The encoder_output = (4,5,8) is ready to go in the Decoder Architecture)

# Decoder Architecture Start

Target Tokens

In [39]:
target_tokens = tf.random.uniform(
    shape = (4,5),
    minval = 0,
    maxval = 100,
    dtype = tf.int32
)

In [40]:
d_model = 8
num_heads = 2
d_ff = 128

Embeddings

In [41]:
target_embedding_layers = tf.keras.layers.Embedding(
    input_dim = 100,
    output_dim = d_model
)

In [42]:
target_embedding = target_embedding_layers(target_tokens)

Positional Encoding

In [43]:
target_pos_encoding = positional_encoding(seq_len,d_model)

In [44]:
decoder_input = target_embedding + target_pos_encoding

In [45]:
print(decoder_input.shape)

(4, 5, 8)


Masked Multihead Attention

In [47]:
masked_multi = tf.keras.layers.MultiHeadAttention(
    num_heads = num_heads,
    key_dim = key_dim
)

In [48]:
masked_attention_output = masked_multi(
    query = decoder_input,
    value = decoder_input,
    key = decoder_input,
    use_causal_mask = True
)

Add & Norm

In [51]:
add_norm1_ = tf.keras.layers.LayerNormalization()

In [52]:
add_norm_input = decoder_input + masked_attention_output

In [53]:
add_norm1_decoder = add_norm1_(add_norm_input)

In [54]:
print(add_norm1_decoder.shape)

(4, 5, 8)


Cross Attention

In [56]:
cross_atte = tf.keras.layers.MultiHeadAttention(
    num_heads = num_heads,
    key_dim = d_model
)

In [57]:
cross_attention = cross_atte(
    query = add_norm1_decoder,
    value = encoder_output,
    key = encoder_output
)

Add & Norm


In [59]:
add_norm2_ = tf.keras.layers.LayerNormalization()

In [60]:
add_norm2_input = add_norm1_decoder + cross_attention

In [61]:
add_norm2_decoder = add_norm2_(add_norm2_input)

In [63]:
print(add_norm2_decoder.shape)

(4, 5, 8)


Feed Forward Neural Network

In [65]:
decoder_fn = tf.keras.Sequential([
    tf.keras.layers.Dense(128,activation="relu"),
    tf.keras.layers.Dense(d_model)
])

In [66]:
ffnn_decoder_output = decoder_fn(add_norm2_decoder)

Add & Norm

In [67]:
add_norm3 = add_norm2_decoder + ffnn_decoder_output

In [68]:
print(add_norm3.shape)

(4, 5, 8)


In [69]:
add_norm3_ = tf.keras.layers.LayerNormalization()

In [70]:
add_norm3_decoder = add_norm3_(add_norm3)

In [71]:
print(add_norm3_decoder.shape)

(4, 5, 8)


Decoder Output

In [72]:
final_decoder_output = add_norm3_decoder

In [73]:
print(final_decoder_output.shape)

(4, 5, 8)


# Decoder Architecture is Complete